In [1]:
from astroquery.sdss import SDSS
import pandas as pd

## Exercise a)

Select galaxies and quasars with redshifts between 0.05 and 0.3 and signal-to-noise ratios greater than 35 near the H $\beta$ line. Ensure that lines [O III] $\lambda$ 5007, H $\beta$ $\lambda$ 4863, and H $\gamma$ $\lambda$ 4341 are present in emission and that the FWHM of H $\beta$ is greater than 1000 km/s.

For each selected spectrum, find the flux ratios of [O III]/H $\beta$, H $\beta$ / H $\gamma$, and [O III]/H $\gamma$, as well as the equivalent width and flux of H $\beta$, redshift, and extinction correction E(B-V) of type SFD 

> **Tip**: the last one can be found in the `galSpecInfo` table).

In [2]:
query_a = """
        SELECT 
        s.specobjid, s.z,
        2.3548 * g.sigma_balmer AS fwhm_hbeta,
        g.h_beta_flux, g.h_beta_eqw,
        g.h_gamma_flux,
        g.oiii_5007_flux, 
        (g.oiii_5007_flux / g.h_beta_flux) AS oiii_hbeta_ratio,
        (g.oiii_5007_flux / g.h_gamma_flux) AS oiii_hgamma_ratio,
        (g.h_beta_flux / g.h_gamma_flux) AS hbeta_hgamma_ratio,
        g2.e_bv_sfd
        
        FROM SpecObj as s
        JOIN GalSpecLine as g ON s.specobjid = g.specobjid
        JOIN GalSpecInfo as g2 ON s.specobjid = g2.specobjid
        
        WHERE
        (s.class = 'GALAXY' OR s.class = 'QSO')
        AND s.z BETWEEN 0.05 AND 0.3
        AND (2.3548 * g.sigma_balmer) > 1000
        AND g.h_beta_flux > 0
        AND g.h_gamma_flux > 0
        AND s.snmedian_g > 35
        """

In [3]:
res_a = SDSS.query_sql(query_a).to_pandas()
res_a.to_csv('tables/exercise_a_results.csv', index=False)

In [4]:
res_a.head()

,specobjid,z,fwhm_hbeta,h_beta_flux,h_beta_eqw,h_gamma_flux,oiii_5007_flux,oiii_hbeta_ratio,oiii_hgamma_ratio,hbeta_hgamma_ratio,e_bv_sfd
0,577616922149939200,0.192200,1177.4,637.22640,-8.170201,396.40620,358.43210,0.562488,0.904204,1.607509,0.027274
1,636238762933774336,0.296773,1177.4,2577.09400,-16.616000,1179.15600,440.13020,0.170785,0.373259,2.185541,0.049731
2,1437865486892165120,0.205535,1177.4,1431.32200,-10.366620,790.96900,858.12100,0.599531,1.084898,1.809580,0.019416
3,2638080301418440704,0.052075,1177.4,78.65923,-0.817247,30.21829,60.32346,0.766896,1.996256,2.603033,0.032096
4,2651523206413838336,0.055108,1177.4,1186.63900,-9.494960,639.15010,1897.62400,1.599159,2.968980,1.856589,0.030219


## Exercise b)

How many objects have you found? Which one of the conditions in the `WHERE` clause is narrowing the results most severely?

> **Tip:** one needs to play with this for a while...

In [5]:
len(res_a)

227

I found 227 objects, let's find the most restrictive conditions...

In [6]:
base_query = """
        SELECT 
        s.specobjid, s.z,
        2.3548 * g.sigma_balmer AS fwhm_hbeta,
        g.h_beta_flux, g.h_beta_eqw,
        g.h_gamma_flux,
        g.oiii_5007_flux, 
        (g.oiii_5007_flux / g.h_beta_flux) AS oiii_hbeta_ratio,
        (g.oiii_5007_flux / g.h_gamma_flux) AS oiii_hgamma_ratio,
        (g.h_beta_flux / g.h_gamma_flux) AS hbeta_hgamma_ratio,
        g2.e_bv_sfd
        
        FROM SpecObj as s
        JOIN GalSpecLine as g ON s.specobjid = g.specobjid
        JOIN GalSpecInfo as g2 ON s.specobjid = g2.specobjid

        WHERE
        (s.class = 'GALAXY' OR s.class = 'QSO')
        AND g.h_beta_flux > 0 
        AND g.h_gamma_flux > 0 
        AND
        """

conditions = ["s.z BETWEEN 0.05 AND 0.3",
              "(2.3548 * g.sigma_balmer) > 1000",
              "s.snmedian_g > 35"]

In [7]:
import itertools
from tqdm import tqdm

results = {}
for i in range(len(conditions)):
    for combo in tqdm(itertools.combinations(conditions, i+1)):
        query = base_query + "\n" + "\nAND ".join(combo)
        #print(query)
        res = SDSS.query_sql(query).to_pandas()
        results[combo] = len(res)

3it [00:00,  4.73it/s]
3it [00:00, 68.49it/s]
1it [00:00, 343.77it/s]


In [8]:
print("Number of objects returned for each combination of conditions:")
for combo, count in results.items():
    print(f"Conditions: {combo} -> Count: {count}")

Number of objects returned for each combination of conditions:
Conditions: ('s.z BETWEEN 0.05 AND 0.3',) -> Count: 500000
Conditions: ('(2.3548 * g.sigma_balmer) > 1000',) -> Count: 42982
Conditions: ('s.snmedian_g > 35',) -> Count: 5106
Conditions: ('s.z BETWEEN 0.05 AND 0.3', '(2.3548 * g.sigma_balmer) > 1000') -> Count: 25822
Conditions: ('s.z BETWEEN 0.05 AND 0.3', 's.snmedian_g > 35') -> Count: 440
Conditions: ('(2.3548 * g.sigma_balmer) > 1000', 's.snmedian_g > 35') -> Count: 1483
Conditions: ('s.z BETWEEN 0.05 AND 0.3', '(2.3548 * g.sigma_balmer) > 1000', 's.snmedian_g > 35') -> Count: 227


From this it seems that the most restrictive condition is the signal to noise-ratio greater than 35

## Exercise c)

Find out if there are any `Subclass` AGN objects satisfying the same conditions as in a). Adapt your code to obtain the result.

In [9]:
query_c = """
        SELECT 
        s.specobjid, s.z, s.class, s.subclass, -- Added subclass here to check for AGN
        2.3548 * g.sigma_balmer AS fwhm_hbeta,
        g.h_beta_flux, g.h_beta_eqw,
        g.h_gamma_flux,
        g.oiii_5007_flux, 
        (g.oiii_5007_flux / g.h_beta_flux) AS oiii_hbeta_ratio,
        (g.oiii_5007_flux / g.h_gamma_flux) AS oiii_hgamma_ratio,
        (g.h_beta_flux / g.h_gamma_flux) AS hbeta_hgamma_ratio,
        g2.e_bv_sfd
        
        FROM SpecObj as s
        JOIN GalSpecLine as g ON s.specobjid = g.specobjid
        JOIN GalSpecInfo as g2 ON s.specobjid = g2.specobjid
        
        WHERE
        (s.class = 'GALAXY' OR s.class = 'QSO')
        AND s.z BETWEEN 0.05 AND 0.3
        AND (2.3548 * g.sigma_balmer) > 1000
        AND g.h_beta_flux > 0
        AND g.h_gamma_flux > 0
        AND s.snmedian_g > 35
        """

In [10]:
res_c = SDSS.query_sql(query_c).to_pandas()
res_c.to_csv('tables/exercise_c_results.csv', index=False)
res_c.head()

,specobjid,z,class,subclass,fwhm_hbeta,h_beta_flux,h_beta_eqw,h_gamma_flux,oiii_5007_flux,oiii_hbeta_ratio,oiii_hgamma_ratio,hbeta_hgamma_ratio,e_bv_sfd
0,1773428476443912192,0.268882,QSO,BROADLINE,1177.400,1758.446000,-17.376420,710.22740,277.13950,0.157605,0.390212,2.475892,0.051156
1,1999766524558075904,0.134188,QSO,BROADLINE,1177.400,1311.827000,-13.298900,560.20080,881.88600,0.672258,1.574232,2.341709,0.031796
2,2595230970286204928,0.193309,GALAXY,NaN,1159.568,523.057100,-3.192120,797.90160,20.48948,0.039173,0.025679,0.655541,0.124201
3,2793445423480596480,0.060622,GALAXY,NaN,1177.400,6.356665,-0.152301,43.33659,24.75229,3.893912,0.571164,0.146681,0.031552
4,2930742271668152320,0.053870,GALAXY,NaN,1173.064,49.934540,-0.370979,31.09171,54.12582,1.083936,1.740844,1.606040,0.025373


In [11]:
# Find if the subclass includes "AGN"
res_c_qso = res_c[res_c['class'] == 'QSO']
res_c_qso['is_agn'] = res_c_qso['subclass'].str.contains('AGN', case=False, na=False)
print(len(res_c_qso.loc[res_c_qso['is_agn']])) 

10


There are only 10 AGN!

## Exercise d)

Modify your solution from a) to include objects with redshift between 0.05 and 0.6. Using this modified solution and the list of objects (`287-plate-mjd-fiber.txt`), submit the SQL query via CrossID.

> **Tip:** You will need to alter the SQL code prepared under a) to fit the requirements of CrossID. Follow the comments you get and be patient.

For this part I used the CrossID tool: https://skyserver.sdss.org/dr18/CrossMatchTools/ObjectCrossID. I followed these steps:

1. Select these options:

<img src="screenshots/options.png" width="800">

2. Upload the table:

<img src="screenshots/template.png" width="800">

3. Edit the default query that manages the cross ID by adding the slections and filters from the query of exercise a

<img src="screenshots/query.png" width="800">

The html output gives the final query that is used:

```
CREATE TABLE #upload ( up_id int, up_plate int, up_mjd int, up_fiber int ) 
INSERT INTO #upload values ( 1, 1949, 53433, 472),( 2, 1273, 52993, 348),( 3, 2030, 53499, 201),( 4, 2520, 54584, 442),( 5, 1940, 53383, 407),
... -- All rows are here 
,( 284, 2774, 54534, 51),( 285, 2485, 54176, 231),( 286, 2292, 53713, 348)
SELECT s.specobjid, s.z, s.ra, s.dec, s.plate, s.mjd, s.fiberid,
(2.3548 * g.sigma_balmer) AS fwhm_hbeta,
g.h_beta_flux, g.h_beta_eqw,
g.h_gamma_flux,
g.oiii_5007_flux, 
(g.oiii_5007_flux / NULLIF(g.h_beta_flux, 0)) AS oiii_hbeta_ratio,
(g.oiii_5007_flux / NULLIF(g.h_gamma_flux, 0)) AS oiii_hgamma_ratio,
(g.h_beta_flux / NULLIF(g.h_gamma_flux, 0)) AS hbeta_hgamma_ratio,
 g2.e_bv_sfd

FROM #upload AS u
JOIN SpecObjAll AS s ON (s.plate = u.up_plate AND s.mjd = u.up_mjd AND s.fiberID = u.up_fiber)
JOIN GalSpecLine AS g ON s.specobjid = g.specobjid
JOIN GalSpecInfo AS g2 ON s.specobjid = g2.specobjid

WHERE
(s.class = "GALAXY" OR s.class = "QSO")
AND s.z BETWEEN 0.05 AND 0.6
AND (2.3548 * g.sigma_balmer) > 1000
AND g.h_beta_flux > 0
AND g.h_gamma_flux > 0
AND s.snmedian_g > 35
```

And the resulting table is this:

In [12]:
res_d = pd.read_csv("tables/Skyserver_CrossID4_23_2026 10_04_54 PM.csv",comment='#')
res_d.head()

,specobjid,z,ra,dec,plate,mjd,fiberid,fwhm_hbeta,h_beta_flux,h_beta_eqw,h_gamma_flux,oiii_5007_flux,oiii_hbeta_ratio,oiii_hgamma_ratio,hbeta_hgamma_ratio,e_bv_sfd
0,717269470510344192,0.599349,315.005190,-7.193428,637,52174,259,1177.4,3992.1940,-11.402230,1681.0080,1007.5240,0.252373,0.599357,2.374881,0.074774
1,462849620679092224,0.532471,44.906114,0.626736,411,51817,381,1177.4,1313.9030,-10.747670,428.4151,552.5841,0.420567,1.289833,3.066892,0.088108
2,1932140511615805440,0.500819,223.843950,12.121759,1716,53827,350,1177.4,798.7177,-9.304626,419.5505,720.5530,0.902137,1.717440,1.903746,0.033267
3,1378205711091132416,0.522838,174.063080,10.575260,1224,52765,379,1177.4,1333.4460,-12.693350,487.0500,453.2127,0.339881,0.930526,2.737802,0.040022
4,2642528650955614208,0.511335,152.627290,25.997088,2347,53757,151,1177.4,1452.4410,-16.707320,670.8977,1001.2380,0.689349,1.492386,2.164921,0.026489


In [13]:
len(res_d)

47

## Exercise e)

Check the spectra of the found objects and download some of them using `wget`.

First I extracted 5 objects plate, mjd and fiberid to search:

In [14]:
plate_mjd_fiber_head = res_d[['plate', 'mjd', 'fiberid']][0:5].to_csv("plate_mjd_fiber_head.csv", index=False,header=False)


Then I copied them into the tool and click on download with ```wget```:

<img src="screenshots/search_spectra.png" width="800">

This gives a txt file with the links to download. With it we can use this command in the terminal to get the fits files:

```wget -i download_url.txt```

## Exercise f) — Bonus

Read the downloaded FITS files and plot the spectra using Python.